In [0]:
%pip install -U databricks-langchain langchain-openai
dbutils.library.restartPython()

INFO: pip is looking at multiple versions of unitycatalog-openai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of unitycatalog-openai[databricks] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 856.1/856.1 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5

In [0]:
from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage

In [0]:
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0
)

In [0]:
@tool
def movie_lookup(movie_name: str):
    """
    Find information about a movie.
    """

    movies = {
        "avatar": {
            "year": 2009,
            "director": "James Cameron",
            "genre": "Sci-Fi"
        },
        "toy story": {
            "year": 1995,
            "director": "John Lasseter",
            "genre": "Animation"
        }
    }

    return movies.get(movie_name.lower(), "Movie not found")

In [0]:
@tool
def book_lookup(book_name: str):
    """
    Find information about a book.
    """

    books = {
        "harry potter": {
            "author": "J.K. Rowling",
            "year": 1997
        },
        "clean code": {
            "author": "Robert Martin",
            "year": 2008
        }
    }

    return books.get(book_name.lower(), "Book not found")

In [0]:
tools = [movie_lookup, book_lookup]

agent = llm.bind_tools(tools)

In [0]:
import pprint
question = "Tell me about Avatar."

response = agent.invoke(
    [HumanMessage(question)]
)

pprint.pprint(response.additional_kwargs)
print (response.content)

{'tool_calls': [{'function': {'arguments': '{ "movie_name": "Avatar" }',
                              'name': 'movie_lookup'},
                 'id': 'call_d0e8d699-8a8a-41db-953e-16fd9ebab007',
                 'type': 'function'}]}


In [0]:
tool_call = response.tool_calls[0]

tool_name = tool_call["name"]

args = tool_call["args"]

print(tool_name)
print(args)

movie_lookup
{'movie_name': 'Avatar'}


In [0]:
result = movie_lookup.invoke(args)

print(result)

{'year': 2009, 'director': 'James Cameron', 'genre': 'Sci-Fi'}


In [0]:
summary = llm.invoke(

f"""
The user asked:

{question}

The tool returned:

{result}

Write a friendly answer.
"""
)

print(summary.content)

You're interested in learning about Avatar. Released in 2009, Avatar is a groundbreaking Sci-Fi film directed by the renowned James Cameron. The movie has received widespread acclaim for its stunning visuals and immersive storyline. If you're a fan of science fiction, you might really enjoy watching Avatar. Would you like to know more about the plot or its reception?
